# Step 3 — Othering Detector

**Goal:** Detect "othering" language in social media posts — language that constructs  
an in-group ("us") in opposition to a dehumanised out-group ("them").

We use rule-based pattern matching across four linguistic families:
1. **Dehumanizing metaphors** — invasion, flood, swarm, infestation, plague…
2. **Moral exclusion** — don't belong, go back, not like us, their kind…
3. **Generalization** — all of them, these people always, they never…
4. **Threat framing** — taking over, replacing us, destroying our…

Each post receives:
- `has_othering` (bool) — at least one family matched
- `othering_score` (int 0-4) — number of distinct families matched
- `matched_patterns` (list) — specific pattern labels that fired

**Input:** `data/processed/dataset_enriched.csv`  
**Output:** `data/processed/dataset_othering.csv`

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from collections import Counter

from othering import detect_othering, apply_othering, ALL_FAMILIES

# Optional: limit rows for quick iteration
MAX_ROWS = None   # set to e.g. 5000 to test faster

## Task 1 — Load enriched dataset

In [ ]:
df = pd.read_csv('../data/processed/dataset_enriched.csv')
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')

if MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
    print(f'Subsampled to {len(df):,} rows')

# Ensure clean_text exists
if 'clean_text' not in df.columns:
    df['clean_text'] = df['text'].fillna('')

df.head(3)

## Task 2 — Apply othering detector

In [ ]:
df = apply_othering(df, text_col='clean_text')

n_oth = df['has_othering'].sum()
print(f'Posts with othering : {n_oth:,} / {len(df):,} ({n_oth/len(df)*100:.1f}%)')

print('\nScore distribution:')
print(df['othering_score'].value_counts().sort_index())

In [ ]:
# Top matched patterns
all_matched = [p for sublist in df['matched_patterns'] for p in sublist]
pattern_counts = Counter(all_matched)

pd.Series(pattern_counts).sort_values(ascending=False).head(20).to_frame('count')

## Task 3 — Validation on 100 random examples

In [ ]:
sample_100 = df.sample(n=min(100, len(df)), random_state=7)

# Proxy ground truth: toxicity > 0.5 AND has_them = True
has_proxy = (
    (sample_100.get('toxicity', pd.Series(0, index=sample_100.index)).fillna(0) > 0.5) &
    (sample_100.get('has_them', pd.Series(False, index=sample_100.index)).fillna(False))
)

predicted = sample_100['has_othering']
tp = (predicted & has_proxy).sum()
fp = (predicted & ~has_proxy).sum()
fn = (~predicted & has_proxy).sum()
tn = (~predicted & ~has_proxy).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print('Proxy ground truth: toxicity > 0.5 AND has_them = True')
print(f'Precision : {precision:.2%}')
print(f'Recall    : {recall:.2%}')
print(f'F1        : {f1:.2%}')
print(f'\nConfusion matrix:')
print(f'             Pred +   Pred -')
print(f'  Actual +    {tp:>4}     {fn:>4}')
print(f'  Actual -    {fp:>4}     {tn:>4}')

In [ ]:
# True positive examples — our detector fires AND proxy agrees
tp_df = sample_100[predicted & has_proxy][['clean_text', 'matched_patterns', 'othering_score']].head(10)
for _, row in tp_df.iterrows():
    print(f"  Patterns: {row['matched_patterns']}")
    print(f"  Text: {str(row['clean_text'])[:150]}")
    print()

In [ ]:
# False positive examples — our detector fires, proxy disagrees
print('--- False positives ---')
fp_df = sample_100[predicted & ~has_proxy][['clean_text', 'matched_patterns']].head(5)
for _, row in fp_df.iterrows():
    print(f"  Patterns: {row['matched_patterns']}")
    print(f"  Text: {str(row['clean_text'])[:150]}")
    print()

## Task 4 — Save dataset_othering.csv

In [ ]:
out_path = '../data/processed/dataset_othering.csv'
df.to_csv(out_path, index=False)
print(f'Saved {len(df):,} rows to {out_path}')
print(f'Columns: {list(df.columns)}')

## Task 5 — Summary & cross-analysis

In [ ]:
print(f"% posts with othering: {df['has_othering'].mean()*100:.1f}%")

if 'source' in df.columns:
    print('\nOthering rate by source:')
    display(df.groupby('source')['has_othering'].mean().mul(100).round(1).to_frame('othering_%'))

if 'toxicity' in df.columns:
    avg_tox_oth = df[df['has_othering']]['toxicity'].mean()
    avg_tox_non = df[~df['has_othering']]['toxicity'].mean()
    print(f'\nAvg toxicity — othering     : {avg_tox_oth:.4f}')
    print(f'Avg toxicity — non-othering : {avg_tox_non:.4f}')

if 'emotion' in df.columns:
    print('\nTop 5 emotions in othering posts:')
    display(df[df['has_othering']]['emotion'].dropna().value_counts().head(5).to_frame())
    print('\nTop 5 emotions in non-othering posts:')
    display(df[~df['has_othering']]['emotion'].dropna().value_counts().head(5).to_frame())

In [ ]:
# Top matched patterns as a table
top_patterns = pd.Series(pattern_counts).sort_values(ascending=False).head(20)
top_patterns_df = top_patterns.to_frame('count')
top_patterns_df['pct_of_posts'] = (top_patterns_df['count'] / len(df) * 100).round(2)
display(top_patterns_df)